## <b><font size="6" color="#EE964B">1. Introduction</font></b><br>

In this project, we are going to analyze a dataset that belongs to a company that sells subscription-based memberships to a physical workspace. Clients choose a `plan_tier` and can add extra services. 

The dataset includes one row per client and it covers four kinds of information:

- **Profile:** age, gender, industry, country, and whether the account is a team account.
- **Contract:** plan tier, days per week included in the plan, membership start and end dates, number of renewals.
- **Behaviour:** total and average weekly visits, entitled vs. actual visits in the last period, last badge swipe, days since the last visit, hours of meeting rooms and event space, guest passes issued.
- **Value:** `lifetime_value`.

The goal of this project is to understand **who our clients are and how they use their subscription**, in order to support decisions on retention and on the commercial offer.

In [ ]:
#libraries
import sqlite3
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from math import ceil
from matplotlib.colors import LinearSegmentedColormap
from dash import Dash, html, dcc, callback, Output, Input
import dash_ag_grid as dag
import plotly.express as px
from itertools import combinations

# for better resolution plots
%config InlineBackend.figure_format = 'retina' # optionally, you can change 'svg' to 'retina'

# Setting seaborn style
main_color = "#40916C"
colors = ["#1B4332", "#40916C", "#95D5B2", "#F4D35E", "#EE964B"]
sns.set_theme(style="ticks", palette=colors)

## <b><font size="6" color="#EE964B">2. Data Exploration and Initial Analysis</font></b><br>

Before cleaning or analyzing relationships, we first need to understand what our dataset contains and how reliable it is. Starting by looking at the first five rows of the dataset.

In [ ]:
df_raw = pd.read_csv("data/acdc_dataset.csv", sep=";", index_col="member_id")
df = df_raw.copy()
df_raw.head()

,age,gender,industry,country,is_team_account,n_colleagues_on_account,plan_tier,plan_days_per_week,membership_start,membership_end,...,uses_event_space,uses_guest_passes,uses_print,uses_mail_handling,uses_phone_booths,uses_day_passes,uses_storage,uses_bike_locker,n_renewals,lifetime_value
member_id,,,,,,,,,,,,,,,,,,,,,
CW000001,38.0,Female,Information Technology,Portugal,False,0,flex,3,2021-08-01,2023-03-06,...,False,True,0,False,True,True,False,False,2,4477.97
CW000002,38.0,Male,Information Technology,Portugal,True,2,HOT_DESK,2,2023-11-24,1970-01-01,...,False,False,0,True,False,False,False,False,0,165.86
CW000003,43.0,Male,Education,DE,True,1,flex,3,2021-12-03,1970-01-01,...,False,False,0,False,False,False,False,False,1,4486.17
CW000004,38.0,F,Legal,portugal,False,0,flex,3,2022-01-02,2022-02-15,...,False,False,0,False,False,False,False,False,0,258.63
CW000005,45.0,Female,Information Technology,United Kingdom,False,0,hot_desk,2,2023-10-24,1970-01-01,...,False,False,0,False,False,True,False,False,0,335.74


By looking at these rows, we can already see some inconsistencies. For example, `gender` has two different ways of representing female (**Female** and **F**), and `plan_tier` has two different ways of writing the same plan (**HOT_DESK** and **hot_desk**).

In the next cell, we use `.info()` to see the number of rows and columns of the dataset, the data type of each column and how many non-null values they have, as this information will be fundamental for many of our later decisions.

In [ ]:
df_raw.info()

<class 'pandas.core.frame.DataFrame'>
Index: 15465 entries, CW000001 to CW015101
Data columns (total 30 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   age                          14907 non-null  float64
 1   gender                       14935 non-null  object 
 2   industry                     15465 non-null  object 
 3   country                      15035 non-null  object 
 4   is_team_account              15465 non-null  bool   
 5   n_colleagues_on_account      15465 non-null  int64  
 6   plan_tier                    15360 non-null  object 
 7   plan_days_per_week           15465 non-null  int64  
 8   membership_start             15465 non-null  object 
 9   membership_end               15465 non-null  object 
 10  last_badge_swipe             15465 non-null  object 
 11  days_since_last_visit        15465 non-null  int64  
 12  total_visits                 15465 non-null  int64  
 13  avg_weekly_

The dataset has a total of 15465 rows and 30 columns originally. We can already see some inconsistencies in the data types: 

- In the first rows of the dataset, we could see that `membership_start` and `membership_end` represent dates, however they are an object type.
- `uses_print` is supposed to be a boolean (like the other columns that start with `uses_`), but its an int type.
- When we checked the first five rows, `lifetime_value`appeared to be a float, and it should be by its description, however its type is object.

We can also see that some columns have null values, so we decided to analyze them in more detail.

In [ ]:
missing = df.isna().sum()
missing_table = pd.DataFrame({
    "missing": missing,
    "percentage of rows": (missing / len(df) * 100).round(2),
})

missing_table = (missing_table[missing_table["missing"] > 0]
                 .sort_values("missing", ascending=False))
print(missing_table)
print(f"Total missing values: {missing_table['missing'].sum()}")
print(f"Total percentage of missing values: {100*(missing_table['missing'].sum())/(df.shape[0]*df.shape[1])}")

                     missing  percentage of rows
meeting_room_hours      1298                8.39
guest_passes_issued     1271                8.22
lifetime_value          1040                6.72
age                      558                3.61
gender                   530                3.43
country                  430                2.78
plan_tier                105                0.68
Total missing values: 5232
Total percentage of missing values: 1.1277077271257678


Seven of the dataset's columns have missing values: the three columns with the highest number of missing values are `meeting_room_hours` (8.39%), `guest_passes_issued` (8.22%) and `lifetime_value` (6.72%). The total number of cells with nulls is 5232, which corresponds to about 1.13% of all cells, therefore the missing data is limited and concentrated in a small number of variables. However, what may matter more is the number of rows affected, since a row with a single missing value may need to be removed or imputed lately.

Next, we used the `.describe()` method to get the descriptive statistics for the three types of columns we have: numeric, boolean and object.

In [ ]:
df_raw.describe().T

,count,mean,std,min,25%,50%,75%,max
age,14907.0,34.858724,10.394649,1.0,29.00,34.00,40.00,149.00
n_colleagues_on_account,15465.0,0.537019,1.177668,-1.0,0.00,0.00,1.00,9.00
plan_days_per_week,15465.0,3.387262,1.458818,0.0,2.00,3.00,5.00,9.00
days_since_last_visit,15465.0,117.109279,275.532807,-956.0,4.00,13.00,55.00,1212.00
total_visits,15465.0,110.406531,182.939340,-112.0,8.00,34.00,132.00,6530.00
avg_weekly_visits,15465.0,1.777214,1.937515,0.0,0.71,1.42,2.41,47.40
entitled_visits_last_period,15465.0,27.517393,13.480235,0.0,17.39,26.09,43.48,78.27
actual_visits_last_period,15465.0,13.800970,11.387207,0.0,5.00,10.00,20.00,60.00
meeting_room_hours,14167.0,33.075570,428.169678,-56.6,0.00,0.00,7.20,41220.00
event_space_hours,15465.0,5.117220,26.013097,-13.5,0.00,0.00,0.00,602.80


- `age`: the minimum age is 1 and the maximum is 149, so this column has invalid values.
- `n_colleagues_on_account`: the median is 0, so at least half of the customers do not have colleagues on the account. The minimum is -1, which tells us that this column also has invalid values.
- The columns `days_since_last_visit`, `total_visits`, `meeting_room_hours`, `event_space_hours`, `guest_passes_issued` and `n_renewals` have minimums below zero, so they also have invalid values. Their distributions are also highly skewed, since their means are significantly larger than their medians.
- The values of `entitled_visits_last_period` are larger than the values of `actual_visits_last_period`, which suggests that customers do not use all the visits they are entitled to.
- `uses_print`: at least half of the values are 0 and at least 25% of the values are 1, so it seems that the only values in this column are 0 and 1. The column appears to be a boolean, as we suspected.

In [ ]:
df_raw.describe(include="bool").T

,count,unique,top,freq
is_team_account,15465,2,False,11112
uses_meeting_rooms,15465,2,False,10202
uses_event_space,15465,2,False,13624
uses_guest_passes,15465,2,False,12651
uses_mail_handling,15465,2,False,11901
uses_phone_booths,15465,2,False,11469
uses_day_passes,15465,2,False,12069
uses_storage,15465,2,False,13169
uses_bike_locker,15465,2,False,15439


The boolean columns have similar conclusions:

- As expected, they only have two unique values (True or False);
- All of them have a count of 15465, which confirms that they don't have missing values;
- The top value for all of them is False, with frequencies ranging from 10202 to 15439, which means that most customers don't use the additional features (almost no one uses the `uses_bike_locker`) and most of the accounts are from individual people, not teams.

In [ ]:
df_raw.describe(include=["object", "string"]).T

,count,unique,top,freq
gender,14935,12,Female,7414
industry,15465,43,Information Technology,3322
country,15035,33,Portugal,9959
plan_tier,15360,19,flex,5182
membership_start,15465,1539,2021-01-01,560
membership_end,15465,1423,1970-01-01,3956
last_badge_swipe,15465,1543,2023-12-30,279
lifetime_value,14425,11860,0,1725


- `gender` has 12 different values, which is more than expected. The top gender is Female which represents about half of the gender non-null values.
- There are 43 unique values in `industry`, with the most common being Information Technology. The number of unique values is high, so we will analyze it later to check for inconsistencies.
- There are *apparently* 33 different countries in our dataset, the most frequent being Portugal which represents about 66% of all rows with country values.
- `plan_tier` should only have 5 unique values but it has 19. This confirms our previous suspicion that this column has inconsistent values.
- Like we said before `membership_start`, `membership_end` and `last_badge_swipe` should be dates and not objects. However, this table shows us that 3956 rows have a `membership_end` of 1970-01-01 that might be a placeholder for when there is no end date.
- `lifetime_value` appears to be a float (its values are numbers and it has 11860 unique values), so we have to change the data type for a better analysis. However, there are 1725 rows (about 12% of non-null rows) with 0 value. Many clients have never generated value. 

## <b><font size="6" color="#EE964B">3. Data Quality Checks</font></b><br>

In this section we are going to analyze on a deeper level the inconsistencies we found in the descriptive statistics and data exploration. Starting by changing `membership_start`, `membership_end` and `last_badge_swipe` to date types and `lifetime_value` to float.

In [ ]:
df['membership_start'] = pd.to_datetime(df["membership_start"], format='mixed')
df['membership_end'] = pd.to_datetime(df["membership_end"], format='mixed')
df['last_badge_swipe'] = pd.to_datetime(df["last_badge_swipe"], format='mixed')

When changing the type of the variable, we realized that some values had symbols that made their type object and not float.

In [ ]:
df["lifetime_value"] = (
    df["lifetime_value"]
    .str.replace("€", "", regex=False)
    .str.replace(",", "", regex=False)
    .str.strip()
    .astype(float)
)

Next, we confirmed that the column `uses_print` only had the values 0 and 1, so we could turn it into a boolean column.

In [ ]:
df_raw["uses_print"].unique()

array([0, 1])

In [ ]:
df["uses_print"] = df["uses_print"].astype("bool")
df["uses_print"].dtype

dtype('bool')

The descriptive statistics of the categorical columns showed a high count of unique values for all variables, so we decided to check them all to look for inconsistencies.

To facilitate the correction of these values, we first put them all in lower case. This corrects errors like the one we saw in `plan_tier` when looking at the first five rows (hot_desk and HOT_DESK).

In [ ]:
# Turn object values to lower case
cols_object = df.select_dtypes(include=["object", "string"]).columns
df[cols_object] = df[cols_object].apply(lambda s: s.str.lower())

# Unique values of each categorical column
for col in ["gender", "country", "plan_tier", "industry"]:
    values = sorted(df[col].dropna().unique())
    print(f"{col}  ({len(values)} unique values, {df[col].isna().sum()} missing)")
    print("   " + ", ".join(values))
    print()

gender  (9 unique values, 530 missing)
    female,  male,  other, f, female, m, male, o, other

country  (24 unique values, 430 missing)
   de, deutschland, es, espana, españa, fr, france, gb, germany, holland, ie, ireland, netherlands, nl, other (europe), portucal, portugal, portugal , pt, spain, the netherlands, u.k., uk, united kingdom

plan_tier  (13 unique values, 105 missing)
   dedicated desk, dedicated_desk, dedicateddesk, flex, full time, full_time, fulltime, hot desk, hot_desk, hotdesk, private office, private_office, privateoffice

industry  (19 unique values, 0 missing)
   consulting, design & creative, design and creative, design/creative, education, finance, health, information technology, it, legal, marketing & pr, marketing and pr, marketing/pr, other, real estate, real-estate, realestate, tech, unknown



- `gender` should only have 3 unique values (female, male and other);
- Multiple countries in `country` have more than one representation, so we should have less than 33 countries;
- `industry` also has inconsistencies so we were correct that 43 unique industries was too high;
- Like we expected `plan_tier` also has inconsistencies

We decided to replace the values with a single standardised value per category, so that each category is counted only once in the analysis.

In [ ]:
#Replacing gender values
df["gender"] = df["gender"].replace([" female", "f"], "female")
df["gender"] = df["gender"].replace([" male", "m"], "male")
df["gender"] = df["gender"].replace([" other", "o"], "other")

#Replacing country values
df["country"] = df["country"].replace({
    "portucal":"portugal", "portugal ":"portugal", "pt":"portugal",
    "de":"germany", "deutschland":"germany",
    "uk":"united kingdom", "u.k.":"united kingdom", "gb":"united kingdom",
    "es":"spain", "españa":"spain", "espana":"spain",
    "nl":"netherlands", "the netherlands":"netherlands", "holland":"netherlands",
    "fr":"france",
    "ie":"ireland"})

#Replacing Plan tier values
df["plan_tier"] = df["plan_tier"].replace(["hotdesk", "hot desk"], "hot_desk")
df["plan_tier"] = df["plan_tier"].replace(["dedicateddesk", "dedicated desk"], "dedicated_desk")
df["plan_tier"] = df["plan_tier"].replace(["fulltime", "full time"], "full_time")
df["plan_tier"] = df["plan_tier"].replace(["privateoffice", "private office"], "private_office")

#Replacing Industry values
df["industry"] = df["industry"].replace(["it", "tech"], "information technology")
df["industry"] = df["industry"].replace(["design & creative", "design/creative"], "design and creative")
df["industry"] = df["industry"].replace(["marketing & pr", "marketing/pr"], "marketing and pr")
df["industry"] = df["industry"].replace(["realestate", "real-estate"], "real estate")

df.describe(include=["object", "string"]).T

,count,unique,top,freq
gender,14935,3,female,7867
industry,15465,11,information technology,3559
country,15035,8,portugal,10508
plan_tier,15360,5,flex,5453


The top values for each category didn't change, but their frequency had a small increase and we have signifficantly less unique values per category.

Next, we analyze if the dataset had duplicates and, if so, we remove them.

In [ ]:
# rows flagged by each method (keep=False marks all copies, not only the later ones)
dup_attrs = df_raw.duplicated(keep=False)        # identical attributes (index ignored)
dup_ids = df_raw.index.duplicated(keep=False)    # repeated ID

print("Repeated ID:", dup_ids.sum())
print("Identical attributes:", dup_attrs.sum())
print("Both:", (dup_ids & dup_attrs).sum())
print("Repeated ID but different attributes:", (dup_ids & ~dup_attrs).sum())
print("Identical attributes but different ID:", (~dup_ids & dup_attrs).sum())

np.int64(295)

In [ ]:
df_nodups = df.drop_duplicates()

295 duplicated rows were succesfully removed!

Lastly, we checked if there were any invalid values.

In [ ]:
df_numeric = df_nodups.select_dtypes(include="number")
neg_counts = (df_numeric < 0).sum()
neg_vars = neg_counts[neg_counts > 0]

print("Numerical variables with negative values:")
display(neg_vars)

Numerical variables with negative values:


n_colleagues_on_account    557
days_since_last_visit       56
total_visits                42
meeting_room_hours          16
event_space_hours            8
guest_passes_issued          7
n_renewals                  65
lifetime_value              81
dtype: int64

Eight columns of the data frame have negative values, `n_colleagues_on_account` being the one with the highest number of negative values.

In [ ]:
(df_nodups["plan_days_per_week"] > 7).sum()

np.int64(53)

53 observations have more than seven plan days per week.

In [ ]:
df_nodups.loc[df_nodups["plan_days_per_week"] > 7, "plan_days_per_week"].value_counts()

plan_days_per_week
9    53
Name: count, dtype: int64

All of the 53 rows with `plan_days_per_week` above 7 have the same value (9).

In [ ]:
(df_nodups["avg_weekly_visits"] > 7).sum()

np.int64(41)

In [ ]:
df_nodups.loc[df_nodups["avg_weekly_visits"] > 7, "avg_weekly_visits"].value_counts()

avg_weekly_visits
35.8    2
19.9    2
26.2    2
26.3    1
10.7    1
41.8    1
7.6     1
44.5    1
42.3    1
25.0    1
43.2    1
29.3    1
28.4    1
8.8     1
8.5     1
27.7    1
19.1    1
38.8    1
27.2    1
10.4    1
15.3    1
8.9     1
10.0    1
43.1    1
42.8    1
27.3    1
47.4    1
26.6    1
36.8    1
46.5    1
7.7     1
8.3     1
31.4    1
27.0    1
20.9    1
21.5    1
43.3    1
10.9    1
Name: count, dtype: int64